# 테스트·린트·빌드와 GitHub Actions

같은 검증 명령을 로컬과 CI에서 실행하고 종료 코드를 확인합니다.

In [1]:
from pathlib import Path
import subprocess, sys
lab = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "ch33-cicd" and (p / "pyproject.toml").is_file()), Path("labs/part7/ch33-cicd")).resolve()
print(sys.version.split()[0], lab.exists())

3.12.13 True


In [2]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=lab, text=True, capture_output=True)
print(result.stdout.strip())
print("exit_code=", result.returncode)
assert result.returncode == 0

.....                                                                    [100%]
5 passed in 0.07s
exit_code= 0


In [3]:
result = subprocess.run(["ruff", "check", "cicd_demo.py", "tests"], cwd=lab, text=True, capture_output=True)
print(result.stdout.strip() or "ruff: PASS")
print("exit_code=", result.returncode)
assert result.returncode == 0

All checks passed!
exit_code= 0


In [4]:
import tempfile
with tempfile.TemporaryDirectory() as out:
    result = subprocess.run([sys.executable, "-m", "build", "--wheel", "--outdir", out], cwd=lab, text=True, capture_output=True)
    wheels = [p.name for p in Path(out).glob("*.whl")]
    print("wheels=", wheels, "exit_code=", result.returncode)
    assert result.returncode == 0 and len(wheels) == 1

wheels= ['ch33_cicd_lab-0.1.0-py3-none-any.whl'] exit_code= 0


## 실패 재현

임시 저장소에서 실패하는 테스트를 추가합니다. 실제 pytest 프로세스가 0이 아닌 종료 코드를 반환합니다.

In [5]:
import tempfile
with tempfile.TemporaryDirectory() as temp:
    p = Path(temp) / "test_regression.py"
    p.write_text("def test_regression():\n    assert 2 + 2 == 5\n")
    failed = subprocess.run([sys.executable, "-m", "pytest", "-q", str(p)], text=True, capture_output=True)
    print(failed.stdout.strip().splitlines()[-1])
    print("exit_code=", failed.returncode)
    assert failed.returncode != 0

1 failed in 0.01s
exit_code= 1


## 수정

같은 테스트의 기대값을 고치면 프로세스가 종료 코드 0을 반환합니다.

In [6]:
with tempfile.TemporaryDirectory() as temp:
    p = Path(temp) / "test_regression.py"
    p.write_text("def test_regression():\n    assert 2 + 2 == 4\n")
    passed = subprocess.run([sys.executable, "-m", "pytest", "-q", str(p)], text=True, capture_output=True)
    print(passed.stdout.strip().splitlines()[-1])
    print("exit_code=", passed.returncode)
    assert passed.returncode == 0

1 passed in 0.00s
exit_code= 0
